# Reproducing Kim et al. (2023), Figure 3, from dandiset 000632

Paper: Kim V, Semenov I, Kiester AS, Keppler MA, Ibey BL, Bixler JN, Colunga Biancatelli RML,
Pakhomov AG (2023). Control of the Electroporation Efficiency of Nanosecond Pulses by Swinging the
Electric Field Vector Direction. Int J Mol Sci 24, 10921. doi:10.3390/ijms241310921.

Dataset: DANDI:000632, "Electroporation efficiency of co-directional and cross-directional paired
pulses", version 0.231012.1930 (doi:10.48324/dandi.000632/0.231012.1930). The paper cites no
version of the dataset (its Data Availability Statement says the data are available on request).
0.231012.1930 is the only published version, and the draft lists the same 24 assets with the same
size, so it is pinned.

Claims: `claims/000632/10.3390-ijms241310921.yaml`, written from the version of record as served by
Europe PMC (PMC10341945), with no supplement and no PDF.

What the dataset holds: 24 NWB files, each one stitched whole-dish image (two channels, MemBrite and
YO-PRO-1) of a BPAE monolayer after four paired pulses in the square electrode array of Figure 3.
It holds none of the triangular-array experiments (Figures 2 and 4) and none of the strobe imaging
of Figure 5.

| claim | tested here |
|---|---|
| fig3-1hz-co-equals-cross | tested |
| fig3-770khz-co-enhanced | tested |
| fig3-770khz-cross-same-as-1hz | tested |
| fig3-770khz-co-above-cross | tested |
| fig3-no-area-dependence | tested (follows fig3-1hz-co-equals-cross) |
| fig2-180-cancellation | blocked, data_missing |
| fig2-crossover-angle | blocked, data_missing |
| fig2-facilitation-plateau | blocked, data_missing |
| fig2-60-fold-co-counter | blocked, data_missing |
| fig4-unipolar-pattern | blocked, data_missing |
| fig5-base-sawtooth | blocked, data_missing |
| fig5-apex-larger-polarization | blocked, data_missing |

Method, in short. For each dish the module `epimg.py` reads the YO-PRO-1 (YP) channel as a reduced
overview, finds the four dark electrode footprints, and takes the array center as the mean of
their centers. It then reads full-resolution pixels for the 200 x 200 um ROI at the center (Methods
4.4) and for a 200 x 200 um ROI at the center of each footprint, where cells were shielded from the
field (Methods 4.4, "To account for 'spontaneous' YP uptake, measurements were also performed
within the hollow electrodes' footprints"). Background reading (a) is the raw mean YP in the center
ROI; reading (b) subtracts the mean of the four footprint ROIs. Groups are compared by two-tailed
Student's t-tests (Figure 3 legend). No step uses random numbers.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import epimg

print("dandiset", epimg.DANDISET, "version", epimg.VERSION)
assets = epimg.list_assets()
print(len(assets), "assets")
t0 = time.time()
# each dish is read in its own process; within it, chunks are fetched in parallel threads
with ProcessPoolExecutor(8, mp_context=mp.get_context("spawn")) as ex:
    dishes = list(ex.map(epimg.measure_dish, assets))
print(f"read {len(dishes)} dishes in {time.time() - t0:.0f} s")

## Data check

In [ ]:
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
rows = []
for d in dishes:
    fp = d["footprints"]
    rows.append(dict(
        subject=d["path"].split("/")[0].removeprefix("sub-"),
        session_id=d["session_id"], direction=d["direction"], freq=d["freq"],
        day=d["session_start_time"][:10], start=d["session_start_time"][11:16],
        shape=str(d["shape"]), unit=d["unit"],
        center_yp=d["center_yp"], fp_yp=np.mean(d["fp_yp"]), fp_yp_min=np.min(d["fp_yp"]),
        fp_yp_max=np.max(d["fp_yp"]), fp_disc_yp=np.mean(d["fp_disc_yp"]),
        center_mb=d["center_mb"], fp_mb=np.mean(d["fp_mb"]),
        fp_radius_um=np.mean([f["radius_um"] for f in fp]),
        parallelogram_dev_um=fp[0]["parallelogram_dev_um"],
        side_um=np.mean([np.hypot((fp[k]["row"] - fp[(k + 1) % 4]["row"]) * epimg.ov_px_um()[0],
                                  (fp[k]["col"] - fp[(k + 1) % 4]["col"]) * epimg.ov_px_um()[1])
                         for k in range(4)]),
        fingerprint=d["chunk_fingerprint"]))
df = pd.DataFrame(rows).sort_values(["direction", "freq", "day", "start"]).reset_index(drop=True)

print("Counts by condition as labelled (session_id) and by experiment day:")
print(pd.crosstab([df.direction, df.freq], df.day, margins=True))
print()
print("session_id matches the condition in the file name for every file:",
      all(s.lower().startswith(dr) and s.endswith(f) for s, dr, f in zip(df.session_id, df.direction, df.freq)))
print("image shapes:", df["shape"].unique(), " data unit:", df.unit.unique())
print()
print("ImageSeries description (identical in all files:",
      len({d["image_description"] for d in dishes}) == 1, "):")
print(" ", dishes[0]["image_description"])
print("Session descriptions, one per condition:")
for s in sorted({d["session_description"] for d in dishes}):
    print(" ", s)

The files hold no pixel size and no channel names beyond the ImageSeries description, which lists
the MemBrite channel first and YO-PRO-1 second. The checks below test both against the images. The
session descriptions say the electrodes were in a triangular array; the images show four footprints
in a square, as the paper describes for Figure 3 (Methods 4.2).

In [ ]:
# Channel order: YP (channel 1) should be high near the array center and low in the shielded
# footprints; MemBrite (channel 0, all cells) should differ much less between the two.
print("center / footprint ratio, median over dishes (range):")
for name, c, f in [("channel 1, YP", df.center_yp, df.fp_yp), ("channel 0, MemBrite", df.center_mb, df.fp_mb)]:
    r = c / f
    print(f"  {name:20s} {r.median():.2f} ({r.min():.2f} to {r.max():.2f})")
r, p = stats.pearsonr(df.center_yp, df.center_mb)
print(f"Pearson r of center YP with center MemBrite across dishes: {r:.2f} (p = {p:.2g})")

# Pixel size: the files give none. Under 0.65 um per pixel (camera pixel 6.5 um, 10x objective),
# the dark footprints should measure 1.5 mm across (Methods 4.2).
print()
print(f"footprint radius under {epimg.UM_PER_PX} um/px: median {df.fp_radius_um.median():.0f} um "
      f"(range {df.fp_radius_um.min():.0f} to {df.fp_radius_um.max():.0f}); nominal outer radius "
      f"{epimg.FOOTPRINT_OD_UM / 2:.0f} um, profile step 50 um")
print(f"array side (center to center of adjacent footprints): median {df.side_um.median():.0f} um "
      f"(range {df.side_um.min():.0f} to {df.side_um.max():.0f})")
print(f"image width under {epimg.UM_PER_PX} um/px: {29696 * epimg.UM_PER_PX / 1000:.1f} mm")
print()
print("Ranges of the variables used (camera counts, unit given as 'n.a.'):")
print(df[["center_yp", "fp_yp", "fp_yp_min", "fp_yp_max", "fp_disc_yp", "center_mb", "fp_mb"]]
      .describe().loc[["min", "50%", "max"]].round(0))

### Files deposited twice

The chunk index of each file (offsets and compressed sizes of all 16384 chunks) is compared between
files, and so are the YP overviews.

In [ ]:
dup = df[df.duplicated("fingerprint", keep=False)]
print("files with identical chunk indexes:")
print(dup[["subject", "session_id", "day", "start"]].to_string())
by_subj = {d["path"].split("/")[0].removeprefix("sub-"): d for d in dishes}
if len(dup):
    a, b = dup.subject.tolist()
    print("YP overviews identical:", np.array_equal(by_subj[a]["overview"], by_subj[b]["overview"]))
    print("start times identical:", by_subj[a]["session_start_time"] == by_subj[b]["session_start_time"])
# the largest correlation between overviews of any other pair, for reference
names = df.subject.tolist()
X = np.array([by_subj[n]["overview"][64:192, 64:192].ravel() for n in names])
Cm = np.corrcoef(X)
np.fill_diagonal(Cm, np.nan)
mask = ~df.subject.isin(dup.subject).values
sub = Cm[np.ix_(mask, mask)]
print(f"largest overview correlation between any two other files: {np.nanmax(sub):.3f}")

### Do the condition labels match the images?

With co-directional pulses one diagonal pair of electrodes carries the field, so uptake should be
stronger along that diagonal than along the other; with cross-directional pulses both diagonals
carry it in turn. The anisotropy index compares, after subtracting the footprint background, mean YP
half-way from the center to the two electrodes of each diagonal: |D1 - D2| / (D1 + D2). This index
is this attempt's check of the labels, not a measure from the paper.

In [ ]:
def aniso(d):
    x = np.array(d["diag_yp"]) - np.mean(d["fp_yp"])
    return abs((x[0] + x[2]) - (x[1] + x[3])) / x.sum()
df["anisotropy"] = [aniso(by_subj[n]) for n in df.subject]
print(df[["subject", "direction", "freq", "day", "anisotropy"]].round(2).to_string())
co = df[(df.direction == "co") & ~df.subject.isin(dup.subject)].anisotropy
cr = df[(df.direction == "cross") & ~df.subject.isin(dup.subject)].anisotropy
u = stats.mannwhitneyu(co, cr, alternative="greater")
print(f"\nwithout the duplicated file: co median {co.median():.2f} (range {co.min():.2f} to {co.max():.2f}, n = {len(co)}), "
      f"cross median {cr.median():.2f} (range {cr.min():.2f} to {cr.max():.2f}, n = {len(cr)}); "
      f"Mann-Whitney U = {u.statistic:.0f}, one-sided p = {u.pvalue:.2g}")
if len(dup):
    print(f"duplicated image: anisotropy {df[df.subject == dup.subject.iloc[0]].anisotropy.iloc[0]:.2f}")

The duplicated image carries two labels. Its start time (2022-11-29, 16:57) falls in the series of
2022-11-29, whose files are numbered 01 to 12 in order of start time and where number 04 is
otherwise missing; the series of 2022-12-04 lacks a number 04 of its own. Its anisotropy is in the
range of the cross-directional dishes. It is counted once, as the cross-directional dish
`BPAE-8bipolar-1Hz-600ns-04`, and `BPAE-8unipolar-1Hz-04` is dropped. The deposit as labelled is shown as a
variant.

In [ ]:
print("Files of each day in order of start time (number from the subject name: condition):")
for day, g in df.sort_values("start").groupby("day"):
    print(" ", day, ", ".join(f"{s[-2:]} {t}: {dr} {f}" for s, t, dr, f in zip(g.subject, g.start, g.direction, g.freq)))
DROP = "BPAE-8unipolar-1Hz-04"
dd = df[df.subject != DROP].copy()
print("\nAfter counting the duplicated recording once:")
print(pd.crosstab([dd.direction, dd.freq], dd.day, margins=True))

In [ ]:
# The located footprints and the center ROI in every dish
fig, axes = plt.subplots(4, 6, figsize=(18, 12.5))
pr, pc = epimg.ov_px_um()
for ax, (_, row) in zip(axes.flat, df.iterrows()):
    d = by_subj[row.subject]
    ov = d["overview"]
    core = ov[64:192, 64:192]
    ax.imshow(ov, cmap="gray", vmin=np.percentile(core, 2), vmax=np.percentile(core, 99))
    for f in d["footprints"]:
        ax.add_patch(plt.Circle((f["col"], f["row"]), f["radius_um"] / pc, fill=False, color="r", lw=0.8))
    rr, cc = np.mean([f["row"] for f in d["footprints"]]), np.mean([f["col"] for f in d["footprints"]])
    ax.add_patch(plt.Rectangle((cc - 100 / pc, rr - 100 / pr), 200 / pc, 200 / pr, fill=False, color="c", lw=1))
    ax.set_xlim(cc - 55, cc + 55); ax.set_ylim(rr + 55, rr - 55)
    ax.set_title(f"{row.subject}\n{row.direction} {row.freq}", fontsize=8)
    ax.axis("off")
plt.suptitle("YP overviews (109 x 116 px blocks, about 71 x 75 um); red: located footprints; cyan: 200 um center ROI", fontsize=10)
plt.tight_layout()
plt.show()
print("Display range per panel: 2nd to 99th percentile of the central overview; each panel shows 7.8 x 8.3 mm "
      "around the array center.")
print("parallelogram deviation of the four footprint centers, um:")
print(df.set_index("subject").parallelogram_dev_um.round(0).to_string())

## The tests of Figure 3

Every test uses the per-dish values below. The MemBrite intensity in the center ROI rises with the
YP intensity across dishes (data check above), so the YP / MemBrite variant divides by a quantity
that itself depends on YP uptake. Center ROIs of 150 and 250 um sides (with the footprint
background), 25% smaller and larger, test the assumed pixel size. The main analysis counts the duplicated recording once
(as cross-directional, 1 Hz). Variants: Welch's test; the footprint background taken over the whole
disc inside 0.6 of the footprint radius; YP divided by MemBrite in the center ROI (variant (b) of
the claims file's opening comment, after background subtraction); without dishes whose footprint
centers are more than 400 um from a parallelogram, where the center ROI may be misplaced; and the
deposit as labelled, with the duplicated image in both 1 Hz groups.

In [ ]:
df["a"] = df.center_yp
df["b"] = df.center_yp - df.fp_yp
df["b_disc"] = df.center_yp - df.fp_disc_yp
df["b_over_mb"] = (df.center_yp - df.fp_yp) / df.center_mb
df["b_roi150"] = [by_subj[n]["center_yp_150"] for n in df.subject] - df.fp_yp
df["b_roi250"] = [by_subj[n]["center_yp_250"] for n in df.subject] - df.fp_yp
dd = df[df.subject != DROP].copy()

def groups(data, col):
    return {(dr, f): data[(data.direction == dr) & (data.freq == f)][col].to_numpy()
            for dr in ("co", "cross") for f in ("1Hz", "770kHz")}

print("Per-group mean +- s.e. (n), main analysis:")
for col in ("a", "b"):
    g = groups(dd, col)
    print(f"  reading ({col}):  " + ";  ".join(
        f"{dr} {f}: {x.mean():.0f} +- {x.std(ddof=1) / np.sqrt(len(x)):.0f} ({len(x)})" for (dr, f), x in g.items()))

def compare(x_key, y_key, title):
    out = []
    variants = [("main", dd, "a"), ("main", dd, "b"),
                ("whole-disc footprint", dd, "b_disc"), ("YP / MemBrite", dd, "b_over_mb"),
                ("ROI 150 um", dd, "b_roi150"), ("ROI 250 um", dd, "b_roi250"),
                ("dev <= 400 um", dd[dd.parallelogram_dev_um <= 400], "a"),
                ("dev <= 400 um", dd[dd.parallelogram_dev_um <= 400], "b"),
                ("as deposited", df, "a"), ("as deposited", df, "b")]
    for label, data, col in variants:
        # "as deposited" uses df, where the duplicated image is under both of its labels
        g = groups(data, col)
        x, y = g[x_key], g[y_key]
        t = stats.ttest_ind(x, y)
        w = stats.ttest_ind(x, y, equal_var=False)
        out.append(dict(variant=label, reading=col, n1=len(x), n2=len(y), mean1=x.mean(), mean2=y.mean(),
                        ratio=x.mean() / y.mean(), diff=x.mean() - y.mean(),
                        t=t.statistic, p_student=t.pvalue, p_welch=w.pvalue))
    res = pd.DataFrame(out)
    print(title)
    print(res.round({"mean1": 3, "mean2": 3, "ratio": 2, "diff": 3, "t": 2, "p_student": 4, "p_welch": 4}).to_string(index=False))
    return res

def by_day(x_key, y_key, col="b"):
    for day, g0 in dd.groupby("day"):
        g = groups(g0, col)
        x, y = g[x_key], g[y_key]
        p = stats.ttest_ind(x, y).pvalue if min(len(x), len(y)) > 1 else np.nan
        print(f"  {day}: reading ({col}) means {x.mean():.0f} (n = {len(x)}) and {y.mean():.0f} (n = {len(y)}), "
              f"ratio {x.mean() / y.mean():.2f}, Student p = {p:.3f}")

def strip(keys, col="b"):
    fig, ax = plt.subplots(figsize=(5, 3.2))
    for i, k in enumerate(keys):
        sub = dd[(dd.direction == k[0]) & (dd.freq == k[1])]
        for day, mk in (("2022-11-29", "o"), ("2022-12-04", "s")):
            v = sub[sub.day == day][col]
            ax.plot(np.full(len(v), i) + (0.08 if mk == "s" else -0.08), v, mk, mfc="none", color="k")
        ax.errorbar(i + 0.25, sub[col].mean(), sub[col].std(ddof=1) / np.sqrt(len(sub)), fmt="_", color="r", ms=14)
    ax.set_xticks(range(len(keys)), [f"{k[0]} {k[1]}" for k in keys])
    ax.set_ylabel(f"YP, reading ({col}), counts"); ax.set_xlim(-0.5, len(keys) - 0.4)
    plt.tight_layout(); plt.show()
    vals = dd[dd.apply(lambda r: (r.direction, r.freq) in keys, axis=1)][col]
    print(f"plotted values range {vals.min():.0f} to {vals.max():.0f}; all points inside the axes; "
          "circles 2022-11-29, squares 2022-12-04, red mean +- s.e.")

## fig3-1hz-co-equals-cross

> At 1 Hz, co-directional and cross-directional (90 degree) paired pulses in the square array have
> the same electroporation efficiency, measured as YP uptake at the array center.

A claim of absence: reproduced when the two-tailed Student's t-test of co- against
cross-directional dishes at 1 Hz gives p >= 0.05 under background reading (b).

In [ ]:
r1 = compare(("co", "1Hz"), ("cross", "1Hz"), "co 1 Hz (1) against cross 1 Hz (2)")
print("\nBy experiment day:")
by_day(("co", "1Hz"), ("cross", "1Hz"))
strip([("co", "1Hz"), ("cross", "1Hz")])

**Result: reproduced.** Under reading (b) the co-directional dishes at 1 Hz (n = 5) average 0.68 of
the cross-directional ones (n = 6), and the Student's t-test gives p = 0.32, so the test does not
find a difference. Reading (a) (ratio 0.75, p = 0.31), Welch's test and every other variant agree
(p from 0.28 to 0.68). The two days disagree: on 2022-11-29 the co-directional mean is 0.45 of the
cross-directional one (p = 0.023, n = 3 and 3), on 2022-12-04 it is 1.43 times it (p = 0.49, n = 2
and 3). The group sizes differ from the paper's n = 6, as one co-directional file is a copy of a
cross-directional one.

## fig3-770khz-co-enhanced

> Raising the pulse frequency from 1 Hz to 770 kHz enhances electroporation by co-directional
> paired pulses, attributed to temporal summation of the induced membrane potential.

Reproduced when the two-tailed Student's t-test of co-directional dishes at 770 kHz against 1 Hz
gives p < 0.05 with the 770 kHz mean higher, under each background reading.

In [ ]:
r2 = compare(("co", "770kHz"), ("co", "1Hz"), "co 770 kHz (1) against co 1 Hz (2)")
print("\nBy experiment day:")
by_day(("co", "770kHz"), ("co", "1Hz"))
strip([("co", "1Hz"), ("co", "770kHz")])

**Result: not reproduced.** The 770 kHz co-directional mean is higher than the 1 Hz one, 1.67-fold
under reading (a) and 1.96-fold under reading (b), but the Student's t-test does not find the
difference (p = 0.15 and 0.16, n = 5 and 5), and no variant reaches p < 0.05 (smallest p = 0.087,
YP / MemBrite). On 2022-11-29 the 770 kHz mean is 2.74 times the 1 Hz one (p = 0.060, n = 3 and 3);
on 2022-12-04 it is 0.76 times it (p = 0.47, n = 2 and 2).

## fig3-770khz-cross-same-as-1hz

> Cross-directional paired pulses at 770 kHz are just as effective as at 1 Hz; charging two
> different regions of the membrane does not cause summation.

A claim of absence: reproduced when the Student's t-test of cross-directional dishes at 770 kHz
against 1 Hz gives p >= 0.05 under reading (b).

In [ ]:
r3 = compare(("cross", "770kHz"), ("cross", "1Hz"), "cross 770 kHz (1) against cross 1 Hz (2)")
print("\nBy experiment day:")
by_day(("cross", "770kHz"), ("cross", "1Hz"))
strip([("cross", "1Hz"), ("cross", "770kHz")])

**Result: reproduced.** Under reading (b) the cross-directional dishes at 770 kHz (n = 7) average
0.84 of those at 1 Hz (n = 6), Student's p = 0.56; the test does not find a difference, and no
variant does (p from 0.37 to 0.82). By day the ratio is 0.71 on 2022-11-29 (p = 0.073) and 1.40 on
2022-12-04 (p = 0.36).

## fig3-770khz-co-above-cross

> At 770 kHz, co-directional paired pulses electroporate more than cross-directional ones.

Reproduced when the Student's t-test of co- against cross-directional dishes at 770 kHz gives
p < 0.05 with the co-directional mean higher, under each background reading.

In [ ]:
r4 = compare(("co", "770kHz"), ("cross", "770kHz"), "co 770 kHz (1) against cross 770 kHz (2)")
print("\nBy experiment day:")
by_day(("co", "770kHz"), ("cross", "770kHz"))
strip([("co", "770kHz"), ("cross", "770kHz")])

**Result: not reproduced.** The co-directional mean at 770 kHz is the higher, 1.45-fold under
reading (a) and 1.59-fold under reading (b), but the Student's t-test does not find the difference
(p = 0.19 and 0.20, n = 5 and 7), nor does any variant (p from 0.19 to 0.52). By day the ratio is
1.75 on 2022-11-29 (p = 0.14) and 0.78 on 2022-12-04 (p = 0.38).

## fig3-no-area-dependence

> Unlike with long pulses, YP uptake after nsEP does not depend on whether the same or a larger area
> of the membrane is electroporated.

Judged by the outcome of fig3-1hz-co-equals-cross. The Figure 2 evidence the Discussion also cites
(fig2-facilitation-plateau) is not in the dataset.

In [ ]:
row = r1[(r1.variant == "main") & (r1.reading == "b")].iloc[0]
print(f"fig3-1hz-co-equals-cross, reading (b): co/cross ratio {row.ratio:.2f}, Student p = {row.p_student:.3f}, "
      f"n = {row.n1} and {row.n2}")
row3 = r3[(r3.variant == "main") & (r3.reading == "b")].iloc[0]
print(f"for reference, cross 770 kHz against cross 1 Hz: ratio {row3.ratio:.2f}, p = {row3.p_student:.3f}")

**Result: reproduced.** It follows fig3-1hz-co-equals-cross, where the test does not find a
difference between co- and cross-directional pulses at 1 Hz (ratio 0.68, p = 0.32).

## Blocked claims

The dataset holds only the square-array experiment of Figure 3. The triangular-array experiments of
Figure 2 (32 ROIs along the bisecting line, single and paired pulses at 1 Hz and 833 kHz) and Figure
4, and the FluoVolt strobe imaging of CHO-K1 cells in Figure 5, are not in it, and the paper says
its data are available on request, so they are not public elsewhere. fig2-180-cancellation,
fig2-crossover-angle, fig2-facilitation-plateau, fig2-60-fold-co-counter, fig4-unipolar-pattern,
fig5-base-sawtooth and fig5-apex-larger-polarization are blocked as data_missing.

## Summary

| claim | outcome |
|---|---|
| fig3-1hz-co-equals-cross | reproduced |
| fig3-770khz-co-enhanced | not reproduced |
| fig3-770khz-cross-same-as-1hz | reproduced |
| fig3-770khz-co-above-cross | not reproduced |
| fig3-no-area-dependence | reproduced |
| fig2-180-cancellation | blocked (data_missing) |
| fig2-crossover-angle | blocked (data_missing) |
| fig2-facilitation-plateau | blocked (data_missing) |
| fig2-60-fold-co-counter | blocked (data_missing) |
| fig4-unipolar-pattern | blocked (data_missing) |
| fig5-base-sawtooth | blocked (data_missing) |
| fig5-apex-larger-polarization | blocked (data_missing) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's. The paper measured intensities with MetaMorph; here the
center ROI is placed automatically at the mean of the four located footprint centers, and the
footprint background is a 200 um ROI at each footprint center, a choice the paper does not state.
The pixel size is not in the files and 0.65 um is assumed (the located footprints measure a median
1.59 mm across under it, against the needles' 1.5 mm). Uptake differs between the two experiment
days (for example co-directional 770 kHz, reading (b), 1835 on 2022-11-29 against 501 on
2022-12-04), and in each of the four comparisons the two days point in opposite directions. The groups have 5, 6, 5 and 7 dishes (co 1 Hz, cross
1 Hz, co 770 kHz, cross 770 kHz) where the paper reports n = 6 for each.

What was found about the dataset.
- One recording is deposited twice under two conditions: `sub-BPAE-8unipolar-1Hz-04`
  (Co-directional 1 Hz) and `sub-BPAE-8bipolar-1Hz-600ns-04` (Cross-directional 1 Hz) have
  identical chunk indexes, images and start times.
- By label, the groups hold 6, 6, 5 and 7 files (co 1 Hz, cross 1 Hz, co 770 kHz, cross 770 kHz),
  not 6 each as in the paper; with the copy counted once, 5, 6, 5 and 7.
- The dataset description and every session description say the electrodes were in a triangular
  array; the images and the paper show a square array of four.
- The files give no pixel size, and the channels are named only in a free-text description.
- The subject names carry undocumented tokens ("8bipolar", "8unipolar", "600ns"); "600ns" marks the
  files of 2022-11-29.
- The dataset holds only Figure 3; Figures 2, 4 and 5 have no public data.